In [66]:
import os 
import json
import torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file

print("Setup Complete. Ready to fetch the weights.")

Setup Complete. Ready to fetch the weights.


In [67]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print("downloading {MODEL_ID} ...")

downloading {MODEL_ID} ...


In [68]:
model_path = snapshot_download(
    repo_id=MODEL_ID, 
    allow_patterns=["*.safetensors", "config.json"]
)
print(f"Model files downloaded to: {model_path}")

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 43240.25it/s]

Model files downloaded to: /Users/sumanthg/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6


In [69]:
config_path = os.path.join(model_path, "config.json")

In [70]:
with open(config_path,"r") as f:
    config=json.load(f)

In [71]:
print(config)

{'architectures': ['LlamaForCausalLM'], 'attention_bias': False, 'bos_token_id': 1, 'eos_token_id': 2, 'hidden_act': 'silu', 'hidden_size': 2048, 'initializer_range': 0.02, 'intermediate_size': 5632, 'max_position_embeddings': 2048, 'model_type': 'llama', 'num_attention_heads': 32, 'num_hidden_layers': 22, 'num_key_value_heads': 4, 'pretraining_tp': 1, 'rms_norm_eps': 1e-05, 'rope_scaling': None, 'rope_theta': 10000.0, 'tie_word_embeddings': False, 'torch_dtype': 'bfloat16', 'transformers_version': '4.35.0', 'use_cache': True, 'vocab_size': 32000}


In [72]:
n_layers = config["num_hidden_layers"]
n_heads = config["num_attention_heads"]
dim = config["hidden_size"]
vocab_size = config["vocab_size"]

print(f"--- Model Blueprint ---")
print(f"Layers: {n_layers}")
print(f"Hidden Dimension: {dim}")
print(f"Attention Heads: {n_heads}")
print(f"Vocabulary Size: {vocab_size}")

--- Model Blueprint ---
Layers: 22
Hidden Dimension: 2048
Attention Heads: 32
Vocabulary Size: 32000


In [73]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"target device = {device}")

target device = mps


In [74]:
weights_file= os.path.join(model_path,"model.safetensors")

In [75]:
state_dict=load_file(weights_file)

In [76]:
print(state_dict)

{'lm_head.weight': tensor([[ 0.0125, -0.0227, -0.0245,  ...,  0.1367, -0.0091,  0.0106],
        [ 0.0099, -0.0239, -0.0259,  ...,  0.1377, -0.0052,  0.0087],
        [ 0.0059,  0.0132, -0.0160,  ..., -0.0030, -0.0056, -0.0327],
        ...,
        [-0.0308, -0.0004,  0.0113,  ...,  0.0654,  0.0152, -0.0064],
        [ 0.0415, -0.0544, -0.0505,  ...,  0.0669, -0.0057,  0.0452],
        [-0.0133, -0.0154, -0.0188,  ...,  0.0762,  0.0124,  0.0100]],
       dtype=torch.bfloat16), 'model.embed_tokens.weight': tensor([[ 6.2883e-06,  4.3511e-06,  4.3213e-06,  ...,  7.0930e-06,
         -4.5300e-06,  9.4175e-06],
        [-1.0910e-03,  1.9302e-03, -1.6632e-03,  ...,  1.9932e-04,
         -6.5231e-04, -4.9973e-04],
        [-1.0376e-03, -4.5471e-03,  5.0781e-02,  ..., -1.0498e-02,
         -1.2741e-03,  7.2021e-03],
        ...,
        [-2.7954e-02, -1.6479e-02,  3.9978e-03,  ...,  1.1597e-02,
         -5.6458e-03, -4.3945e-03],
        [-9.4604e-03, -2.9297e-03,  3.5095e-03,  ..., -5.4626e-

In [77]:
model_weights = {}
for name, tensor in state_dict.items():
    model_weights[name] = tensor.to(device=device, dtype=torch.float16)
print(f"Successfully loaded {len(model_weights)} tensors to {device}.")

Successfully loaded 201 tensors to mps.


In [78]:
embed_key = "model.embed_tokens.weight"

In [79]:
if embed_key in model_weights:
    weight = model_weights[embed_key]
    print(f"Found Embedding Table!")
    print(f"Shape: {weight.shape}") # Should be [32000, 2048]
    print(f"Device: {weight.device}") # Should be mps:0
    print(f"First few values of the first token: {weight[0][:5]}")
else:
    print("Error: Could not find embedding weights. Check the key names.")

Found Embedding Table!
Shape: torch.Size([32000, 2048])
Device: mps:0
First few values of the first token: tensor([6.3181e-06, 4.3511e-06, 4.2915e-06, 9.6560e-06, 4.7684e-06],
       device='mps:0', dtype=torch.float16)


In [80]:
# Check the length of that specific row
row_length = len(model_weights["model.embed_tokens.weight"][0])
print(f"Total numbers in this token's vector: {row_length}")

Total numbers in this token's vector: 2048


In [81]:
from sentencepiece import SentencePieceProcessor
from huggingface_hub import hf_hub_download


In [82]:
tokenizer_path = hf_hub_download(repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0", filename="tokenizer.model")

In [83]:
sp_model = SentencePieceProcessor()
sp_model.load(tokenizer_path)

print(f"Tokenizer loaded. Vocabulary size: {sp_model.get_piece_size()}")
print(f"Beginning of String (BOS) ID: {sp_model.bos_id()}")

Tokenizer loaded. Vocabulary size: 32000
Beginning of String (BOS) ID: 1


In [84]:
prompt = "Building an inference engine is"
token_ids = sp_model.encode(prompt,add_bos=True, add_eos=False)
print(f"Token IDs for prompt: {token_ids}")

Token IDs for prompt: [1, 17166, 385, 27262, 6012, 338]


In [85]:
pieces = [sp_model.id_to_piece(idx) for idx in token_ids]
print(f"Original: {prompt}")
print(f"Token IDs: {token_ids}")
print(f"Sub-word Pieces: {pieces}")

Original: Building an inference engine is
Token IDs: [1, 17166, 385, 27262, 6012, 338]
Sub-word Pieces: ['<s>', '▁Building', '▁an', '▁inference', '▁engine', '▁is']


In [86]:
input_ids = torch.tensor([token_ids]).to(device)

In [87]:
embedding_table=model_weights["model.embed_tokens.weight"]
print(embedding_table.shape)

torch.Size([32000, 2048])


In [88]:
input_embeddings = torch.embedding(embedding_table, input_ids)

In [89]:
print(f"Input IDs Shape: {input_ids.shape}") 
print(f"Final Embedding Shape: {input_embeddings.shape}")

Input IDs Shape: torch.Size([1, 6])
Final Embedding Shape: torch.Size([1, 6, 2048])


In [90]:
#day 3
import torch

def precompute_rope_frequencies(dim:int,seq_len:int,theta:float=10000.0):
    powers = torch.arange(0, dim, 2)[: (dim // 2)].float() / dim
    freqs = 1.0 / (theta ** powers)
    # Create the 'time' steps (0, 1, 2, ... seq_len)
    t = torch.arange(seq_len)
    
    # Outer product: Multiply each time step by each frequency
    # result shape: [seq_len, dim // 2]
    freqs = torch.outer(t, freqs)
    
    # Convert to polar coordinates (complex numbers)
    # This represents the 'Rotation' on a 2D circle
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis

head_dim = 64 
max_seq_len = 2048 # The max memory of the model
freqs_cis = precompute_rope_frequencies(head_dim, max_seq_len).to("mps")

print(f"RoPE Frequency Table Shape: {freqs_cis.shape}")


RoPE Frequency Table Shape: torch.Size([2048, 32])


In [91]:
# # Cell 10: Applying the Rotation (Fixed for Multi-Head & GQA)
def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor):
    # x shape: [Batch, Heads, Seq_Len, Head_Dim]
    # 1. Reshape x to treat every 2 numbers as a complex number
    # We use -1 to automatically handle whatever the Head_Dim is
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    
    # 2. Align frequencies to match x_complex
    # We need freqs_cis to be [1, 1, Seq_Len, Dim] so it 'broadcasts' 
    # to every Batch and every Head simultaneously.
    # We pull the actual Seq_Len from x.shape[2]
    target_seq_len = x.shape[2]
    freqs_cis_reshaped = freqs_cis[:target_seq_len].view(1, 1, target_seq_len, -1)
    
    # 3. Multiply! (Rotation) and convert back to real numbers
    # Multiplication in complex space = Rotation in 2D space
    x_rotated = torch.view_as_real(x_complex * freqs_cis_reshaped).flatten(3)
    
    return x_rotated.type_as(x)

# --- Verification with TinyLlama Shapes ---
# Test 1: Query Heads (32 heads)
sample_q = torch.randn(1, 32, 8, 64).to(device)
rotated_q = apply_rotary_emb(sample_q, freqs_cis)

# Test 2: Key Heads (4 heads)
sample_k = torch.randn(1, 4, 8, 64).to(device)
rotated_k = apply_rotary_emb(sample_k, freqs_cis)

print(f"Query Rotation Successful: {rotated_q.shape}") # Should be [1, 32, 8, 64]
print(f"Key Rotation Successful:   {rotated_k.shape}") # Should be [1, 4, 8, 64]

Query Rotation Successful: torch.Size([1, 32, 8, 64])
Key Rotation Successful:   torch.Size([1, 4, 8, 64])


In [92]:
# Cell 11: Verification
# If we rotate at position 0, it shouldn't change much (angle = 0)
# If we rotate at position 7, it should be different.

diff_pos_0 = torch.norm(sample_query[0, 0] - rotated_query[0, 0])
diff_pos_7 = torch.norm(sample_query[0, 7] - rotated_query[0, 7])

print(f"Change at Position 0: {diff_pos_0.item():.6f}") # Should be near 0
print(f"Change at Position 7: {diff_pos_7.item():.6f}") # Should be much larger

Change at Position 0: 0.000000
Change at Position 7: 5.339625


In [93]:
# Check the first 10 keys to see the naming pattern
print("Here are some actual keys in your model_weights:")
print(list(model_weights.keys())[:15])

Here are some actual keys in your model_weights:
['lm_head.weight', 'model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight']


In [94]:
# Cell 12: Multi-Head Self-Attention
import torch.nn.functional as F

# 1. Setup dimensions
n_heads_q = 32
n_heads_kv = 4 
head_dim = 64
batch_size, seq_len, _ = input_embeddings.shape

# 2. Linear Projections (Creating Q, K, and V)
# We multiply our embeddings by the weight matrices from Day 1
q = (input_embeddings @ model_weights["model.layers.0.self_attn.q_proj.weight"].T)
k = (input_embeddings @ model_weights["model.layers.0.self_attn.k_proj.weight"].T)
v = (input_embeddings @ model_weights["model.layers.0.self_attn.v_proj.weight"].T)

# 3. Reshape for Multi-Head Processing
# Shape: [Batch, Seq, Heads, Dim] -> [Batch, Heads, Seq, Dim]
q = q.view(batch_size, seq_len, n_heads_q, head_dim).transpose(1, 2) 
k = k.view(batch_size, seq_len, n_heads_kv, head_dim).transpose(1, 2) 
v = v.view(batch_size, seq_len, n_heads_kv, head_dim).transpose(1, 2)

# 4. Apply RoPE (The Position "Twist" from Day 3)
q = apply_rotary_emb(q, freqs_cis[:seq_len])
k = apply_rotary_emb(k, freqs_cis[:seq_len])

# 5. Grouped-Query Attention: Repeat K and V 8 times to match the 32 Q heads
k = k.repeat_interleave(n_heads_q // n_heads_kv, dim=1) 
v = v.repeat_interleave(n_heads_q // n_heads_kv, dim=1) 

# 6. Scaled Dot-Product Attention
# Q * K^T / sqrt(64)
scores = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
weights = F.softmax(scores.float(), dim=-1).type_as(q)

# 7. Merge Values and Project Output
# Combine the weights with V and squash back to 2048 dimensions
attention_output = torch.matmul(weights, v).transpose(1, 2).reshape(batch_size, seq_len, 2048)
final_attention_output = attention_output @ model_weights["model.layers.0.self_attn.o_proj.weight"].T

print(f"Day 4 Complete! Attention Matrix Shape: {weights.shape}")

Day 4 Complete! Attention Matrix Shape: torch.Size([1, 32, 6, 6])


In [95]:
# Cell 13: Feed-Forward Network and Output Head

# 1. Internal RMSNorm (Keeping the signal clean)
def rms_norm(x, weight):
    return (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6)) * weight

# Apply norm before the MLP
norm_output = rms_norm(final_attention_output, model_weights["model.layers.0.post_attention_layernorm.weight"])

# 2. The MLP (Gate, Up, and Down Projections)
# Llama uses the SwiGLU activation: (Silu(w1) * w3) @ w2
w1 = model_weights["model.layers.0.mlp.gate_proj.weight"]
w2 = model_weights["model.layers.0.mlp.down_proj.weight"]
w3 = model_weights["model.layers.0.mlp.up_proj.weight"]

ffn_gate = F.silu(norm_output @ w1.T) 
ffn_up = norm_output @ w3.T
ffn_output = (ffn_gate * ffn_up) @ w2.T

# 3. Final Norm and Logits
# We only care about the last token in the sequence for prediction
last_token_vector = ffn_output[:, -1, :] 
final_norm_output = rms_norm(last_token_vector, model_weights["model.norm.weight"])

# Multiply by the Output Head to get scores for all 32,000 words
logits = final_norm_output @ model_weights["lm_head.weight"].T

# 4. Decode the Result
next_token_id = torch.argmax(logits, dim=-1).item()
predicted_piece = sp_model.id_to_piece(next_token_id)

print(f"--- Day 5 Results ---")
print(f"Predicted Token ID: {next_token_id}")
print(f"Predicted Word: '{predicted_piece}'")

--- Day 5 Results ---
Predicted Token ID: 7420
Predicted Word: 'lain'


In [2]:
# # DAY 6: PHYSICAL BLOCK MANAGER

class PhysicalBlockManager:
    def __init__(self, total_gpu_memory_gb, block_size_mb=16):
        # 1. Setup the "Warehouse"
        # We'll assume a 16GB M2 Air
        # Reserve 5GB for Model Weights, 1GB for buffers
        available_memory_gb = total_gpu_memory_gb - 5 - 1
        available_memory_mb = available_memory_gb * 1024
        
        self.block_size_mb = block_size_mb
        self.num_total_blocks = int(available_memory_mb // block_size_mb)
        
        # The "Free Pool": A list of all available block IDs [0, 1, 2... N]
        self.free_blocks = list(range(self.num_total_blocks))
        
        # Tracking: Which blocks belong to which request?
        # { "request_id": [block_id, block_id, ...] }
        self.active_allocations = {}
        
        print(f"Initialized Block Manager with {self.num_total_blocks} blocks.")

    def allocate(self, request_id, num_blocks_needed):
        """Allocates a specific number of blocks to a request."""
        if len(self.free_blocks) < num_blocks_needed:
            print(f"OOM! Requested {num_blocks_needed}, but only {len(self.free_blocks)} left.")
            return None
        
        allocated_ids = []
        for _ in range(num_blocks_needed):
            # Pop from the list (O(1) speed)
            block_id = self.free_blocks.pop(0)
            allocated_ids.append(block_id)
            
        self.active_allocations[request_id] = allocated_ids
        return allocated_ids

    def free(self, request_id):
        """Reclaims blocks when a request is finished."""
        if request_id in self.active_allocations:
            blocks_to_return = self.active_allocations.pop(request_id)
            self.free_blocks.extend(blocks_to_return)
            print(f"Reclaimed {len(blocks_to_return)} blocks from {request_id}.")
        else:
            print("Request ID not found.")

# --- DAY 6 SUCCESS CRITERION TEST ---

# 1. Initialize with 16GB
manager = PhysicalBlockManager(total_gpu_memory_gb=16)

# 2. Request 1 needs 5 blocks
req1_blocks = manager.allocate("Request_1", 5)
print(f"Request 1 Blocks: {req1_blocks}")

# 3. Request 2 needs 3 blocks
req2_blocks = manager.allocate("Request_2", 3)
print(f"Request 2 Blocks: {req2_blocks}")
print(f"Free blocks remaining: {len(manager.free_blocks)}")

# 4. Free Request 1 (Simulating a user closing a tab)
manager.free("Request_1")
print(f"Free blocks after reclaiming: {len(manager.free_blocks)}")

# 5. Request 3 can now reuse the blocks Request 1 just left behind!
req3_blocks = manager.allocate("Request_3", 5)
print(f"Request 3 Blocks: {req3_blocks} (Notice they are the same IDs as Req 1!)")

Initialized Block Manager with 640 blocks.
Request 1 Blocks: [0, 1, 2, 3, 4]
Request 2 Blocks: [5, 6, 7]
Free blocks remaining: 632
Reclaimed 5 blocks from Request_1.
Free blocks after reclaiming: 637
Request 3 Blocks: [8, 9, 10, 11, 12] (Notice they are the same IDs as Req 1!)
